Data Preparation

In [1]:
import pandas as pd

df = pd.read_csv("data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.shape

(7043, 21)

In [2]:
df.describe(include="all")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [3]:
from numpy import int32, int64

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
for col in df.columns:
    if df[col].nunique() == 2 and set(df[col].unique()) not in {0, 1}:
        df[col] = df[col].map({'Yes': 1,
                                 'No': 0,
                                 'Male': 1,
                                 'Female': 0,
                     }).fillna(0).astype(dtype=int64)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,0,0,1,0,1,0,No phone service,DSL,No,...,No,No,No,No,Month-to-month,1,Electronic check,29.85,29.85,0
1,5575-GNVDE,1,0,0,0,34,1,No,DSL,Yes,...,Yes,No,No,No,One year,0,Mailed check,56.95,1889.50,0
2,3668-QPYBK,1,0,0,0,2,1,No,DSL,Yes,...,No,No,No,No,Month-to-month,1,Mailed check,53.85,108.15,1
3,7795-CFOCW,1,0,0,0,45,0,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,0,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,0,0,0,0,2,1,No,Fiber optic,No,...,No,No,No,No,Month-to-month,1,Electronic check,70.70,151.65,1


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder


df = df.drop("customerID", axis=1)
df.head()
multi_cat_cols = [ col for col in df.columns if df[col].nunique() > 2 and df[col].nunique() < 5 ]
preprocessor = ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), multi_cat_cols)], remainder='passthrough')
#df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True)
#df.info()

Model Training - Random Forest

In [5]:
import mlflow
import mlflow.models
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score
import optuna
import time

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

#Initiate MLflow
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment(f"S25021301_COM763_{time.strftime("%Y-%m-%d %H:%M:%S")}")

#Extract the results (y) and split the data into test and train
y = df["Churn"]
X = df.drop("Churn", axis=1)

random_state = 36
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=random_state)

with mlflow.start_run(run_name="Random Forest"):
    #Define the Optuna objective method to optimize the hyperparameters
    def objective(trial: optuna.Trial):
        params = {
           "n_estimators": trial.suggest_int("n_estimators", 100, 600),
            "max_depth": trial.suggest_int("max_depth",3, 15),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
            "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
            "random_state": random_state,
            "n_jobs": -1, 
            "class_weight": trial.suggest_categorical("class_weight", ["balanced", "balanced_subsample"]) 
        }
        base_model = RandomForestClassifier(**params)
        model = Pipeline(steps=[('preprocessor', preprocessor),
                                ('classifier', base_model)]) 
        scores = cross_val_score(model, X, y, cv=3, scoring="recall")
        return scores.mean()

    #Run 30 Optuna trials to identify the best hyperparameters
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=30)
    best_params = study.best_params

    #Train the model with the best params identified
    base_model = RandomForestClassifier(**best_params, random_state=36, n_jobs=-1)
    model = Pipeline(steps=[('preprocessor', preprocessor),
                            ('classifier', base_model)])
    t_start = time.time()
    model.fit(X_train, y_train)
    t_taken = time.time() - t_start

    #Test the model
    p_start = time.time() 
    y_preds = model.predict(X_test)
    p_taken = time.time() - p_start

    #Measure Accuracy, Recall and F1-score
    accuracy = accuracy_score(y_test, y_preds)
    recall = recall_score(y_test, y_preds)
    f1 = f1_score(y_test, y_preds)
    signature = mlflow.models.infer_signature(X_train, model.predict(X_train))

    #Save the model and document in mlflow
    mlflow.log_param("model", "Random Forest") 
    mlflow.log_params(best_params)
    mlflow.log_metric("train_time", t_taken)
    mlflow.log_metric("inference_time", p_taken)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1-score", f1)
    mlflow.sklearn.log_model(model, name="model", signature=signature)


2026/07/05 18:20:32 INFO mlflow.tracking.fluent: Experiment with name 'S25021301_COM763_2026-07-05 18:20:32' does not exist. Creating a new experiment.
[I 2026-07-05 18:20:33,030] A new study created in memory with name: no-name-323f07c7-2a3b-4958-b896-c004de7f3153
[I 2026-07-05 18:20:33,783] Trial 0 finished with value: 0.8073836276083467 and parameters: {'n_estimators': 103, 'max_depth': 5, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 'log2', 'criterion': 'entropy', 'class_weight': 'balanced'}. Best is trial 0 with value: 0.8073836276083467.
[I 2026-07-05 18:20:36,121] Trial 1 finished with value: 0.78705189941145 and parameters: {'n_estimators': 327, 'max_depth': 9, 'min_samples_split': 7, 'min_samples_leaf': 3, 'max_features': 'log2', 'criterion': 'entropy', 'class_weight': 'balanced'}. Best is trial 0 with value: 0.8073836276083467.
[I 2026-07-05 18:20:40,705] Trial 2 finished with value: 0.7619047619047619 and parameters: {'n_estimators': 548, 'max_depth': 9, '

🏃 View run Random Forest at: http://localhost:5000/#/experiments/12/runs/ffdd931b27d74560ba6b644f3f3a2ed9
🧪 View experiment at: http://localhost:5000/#/experiments/12


Model Training - XGBoost

In [6]:
import mlflow.sklearn
from xgboost import XGBClassifier

with mlflow.start_run(run_name="XGBoost"):
    #Define the Optuna objective method to fine-tune hyperparameters to XGBoost
    def objective(trial: optuna.Trial):
       params = {
            "n_estimators": trial.suggest_int("n_estimators", 300, 800),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "subsample": trial.suggest_float("subsample", 0.5, 1),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "scale_pos_weight": (y==0).sum() / (y==1).sum(),
            "random_state": random_state,
            "n_jobs": -1,
            "eval_metric": "logloss"
        } 
       
       base_model = XGBClassifier(**params)
       model = Pipeline(steps=[('preprocessor', preprocessor),
                                ('classifier', base_model)]) 
       scores = cross_val_score(model, X_train, y_train, cv=3, scoring="recall")
       return scores.mean()

    study.optimize(objective, n_trials=30)
    best_params = study.best_params
    
    #Train the model with the best params identified
    base_model = XGBClassifier(**best_params, random_state=random_state, n_jobs=-1, eval_metric="logloss", scale_pos_weight=(y==0).sum() / (y==1).sum())
    model = Pipeline(steps=[('preprocessor', preprocessor),
                                ('classifier', base_model)])
    t_start = time.time()
    model.fit(X_train, y_train)
    t_taken = time.time() - t_start

    #Test the model
    p_start = time.time() 
    y_preds = model.predict(X_test)
    p_taken = time.time() - p_start

    #Measure Accuracy, Recall and F1-score
    accuracy = accuracy_score(y_test, y_preds)
    recall = recall_score(y_test, y_preds)
    f1 = f1_score(y_test, y_preds)
    signature = mlflow.models.infer_signature(X_train, model.predict(X_train))


    #Save the model and document in mlflow
    mlflow.log_param("model", "XGBoost") 
    mlflow.log_params(best_params)
    mlflow.log_metric("train_time", t_taken)
    mlflow.log_metric("inference_time", p_taken)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1-score", f1)
    mlflow.sklearn.log_model(model, name="model", signature=signature, skops_trusted_types=['xgboost.core.Booster', 'xgboost.sklearn.XGBClassifier'])


[I 2026-07-05 18:21:43,684] Trial 30 finished with value: 0.622329122366372 and parameters: {'n_estimators': 684, 'learning_rate': 0.13356724992498858, 'max_depth': 4, 'subsample': 0.8224545284414273, 'colsample_bytree': 0.6610100782587797}. Best is trial 13 with value: 0.815944355270198.
[I 2026-07-05 18:21:45,017] Trial 31 finished with value: 0.7915660043351319 and parameters: {'n_estimators': 344, 'learning_rate': 0.010035816345946094, 'max_depth': 3, 'subsample': 0.5172092871791631, 'colsample_bytree': 0.9916376606488311}. Best is trial 13 with value: 0.815944355270198.
[I 2026-07-05 18:21:45,897] Trial 32 finished with value: 0.7130919565919194 and parameters: {'n_estimators': 377, 'learning_rate': 0.19563507675316033, 'max_depth': 3, 'subsample': 0.9887496055232836, 'colsample_bytree': 0.5030329710540803}. Best is trial 13 with value: 0.815944355270198.
[I 2026-07-05 18:21:46,923] Trial 33 finished with value: 0.7461766051872586 and parameters: {'n_estimators': 329, 'learning_ra

🏃 View run XGBoost at: http://localhost:5000/#/experiments/12/runs/f6bbd385b13242f4bbb2568d58fbbf67
🧪 View experiment at: http://localhost:5000/#/experiments/12


Model Training - LightGBM

In [ ]:
from tabnanny import verbose

import lightgbm
import mlflow.lightgbm
import mlflow.lightgbm

with mlflow.start_run(run_name="LightGBM"):
    
    #Define the Optuna objective method to fine-tune hyperparameters to LightGBM
    def objective(trial: optuna.Trial):
       params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 300),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "subsample": trial.suggest_float("subsample", 0.5, 1),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "scale_pos_weight": (y==0).sum() / (y==1).sum(),
            "random_state": random_state,
            "n_jobs": -1,
            "objective": "binary",
            "metric": "binary_logloss",
            "verbose": -1 
        } 
       
       base_model = lightgbm.LGBMClassifier(**params)
       model = Pipeline(steps=[('preprocessor', preprocessor),
                                ('classifier', base_model)])

       scores = cross_val_score(model, X_train, y_train, cv=3, scoring="recall")
       return scores.mean()

    study.optimize(objective, n_trials=30)
    best_params = study.best_params
    
    #Train the model with the best params identified
    base_model = lightgbm.LGBMClassifier(**best_params, n_jobs=-1, objective="binary", metric="binary_logloss", verbose=-1)
    model = Pipeline(steps=[('preprocessor', preprocessor),
                                ('classifier', base_model)])

    t_start = time.time()
    model.fit(X_train, y_train)
    t_taken = time.time() - t_start

    #Test the model
    p_start = time.time() 
    y_preds = model.predict(X_test)
    p_taken = time.time() - p_start

    #Measure Accuracy, Recall and F1-score
    accuracy = accuracy_score(y_test, y_preds)
    recall = recall_score(y_test, y_preds)
    f1 = f1_score(y_test, y_preds)
    signature = mlflow.models.infer_signature(X_train, model.predict(X_train))

    #Save the model and document in mlflow
    mlflow.log_param("model", "LightGBM") 
    mlflow.log_params(best_params)
    mlflow.log_metric("train_time", t_taken)
    mlflow.log_metric("inference_time", p_taken)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1-score", f1)
    mlflow.sklearn.log_model(model, name="model", signature=signature, skops_trusted_types=['collections.OrderedDict', 'lightgbm.basic.Booster', 'lightgbm.sklearn.LGBMClassifier'])

/home/lahiruk/Workspace/S25021301_COM763/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/lahiruk/Workspace/S25021301_COM763/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/lahiruk/Workspace/S25021301_COM763/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-07-05 18:22:14,243] Trial 60 finished with value: 0.7361671331722736 and parameters: {'n_estimators': 210, 'learning_rate': 0.18442570370447836, 'max_depth': 3, 'subsample': 0.6661947078035436, 'colsample_bytree': 0.9498995047389716}. Best is trial 13 with value: 0.815944355270198.
/home/lahiruk/Workspace/S250213

🏃 View run LightGBM at: http://localhost:5000/#/experiments/12/runs/420e1a5df4214783bbaf18aa8b4ba986
🧪 View experiment at: http://localhost:5000/#/experiments/12


MlflowException: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['collections.OrderedDict', 'lightgbm.basic.Booster', 'lightgbm.sklearn.LGBMClassifier'].